In [ ]:
!ls /kaggle/input/
# If the dataset you attached via "Add Data" isn't named exactly
# marta-debate-coverage-pkg, the folder name printed above is the real one —
# swap it into the cp line below before re-running.

!rm -rf /kaggle/working/marta
!cp -r /kaggle/input/marta-debate-coverage-pkg /kaggle/working/marta
%cd /kaggle/working/marta
!mkdir -p data/debates

In [3]:
!pip install -q langchain-ollama langchain-core langgraph pydantic rich transformers arch pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.3/981.3 kB 21.4 MB/s eta 0:00:0000:01


In [ ]:
# Diagnostic only — TechnicalAgent silently degrades to momentum-only mode if this
# fails, so it won't crash the run, but it does mean losing the trained TFT encoder's
# attention-weight signal for every debate. Kaggle GPU images normally ship torch;
# if this prints an error, report it back before the real run.
import torch
print(torch.__version__, "cuda available:", torch.cuda.is_available())

In [ ]:
# Required — agents/sentiment_agent.py only checks this local path; it does NOT fall
# back to pulling from Hugging Face on its own. Skipping this makes SentimentAgent
# silently return "neutral" for every debate with no error.
from huggingface_hub import snapshot_download
snapshot_download("ProsusAI/finbert", local_dir="/kaggle/working/marta/models/finbert")

In [ ]:
!apt-get -qq update && apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time, requests

ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("/kaggle/working/ollama.log", "w"),
    stderr=subprocess.STDOUT,
)

for _ in range(30):
    try:
        requests.get("http://localhost:11434")
        print("Ollama is up")
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("Ollama didn't come up — check /kaggle/working/ollama.log")

!ollama pull mistral:7b-instruct

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/marta")

from utils.freshness import CRYPTO5_DEBATE_ASSETS, CRYPTO5_PROCESSED, load_debate_data
regime_stream, aligned_dfs = load_debate_data(processed_dir=CRYPTO5_PROCESSED, assets=CRYPTO5_DEBATE_ASSETS)
assert len(aligned_dfs) == 5, f"expected 5 assets, got {list(aligned_dfs.keys())}"
print("Data OK:", len(regime_stream), "regime rows,", list(aligned_dfs.keys()))

# One real debate, timed — confirms the full agent chain + Ollama + FinBERT work
# end-to-end before the long run, and gives a real per-debate time estimate.
import time
sys.path.insert(0, "/kaggle/working/marta/scripts")
from kaggle_debate_coverage import _select_timestamps
from agents.macro_agent import MacroAgent
from agents.technical_agent import TechnicalAgent
from agents.risk_agent import RiskAgent
from agents.fund_manager import FundManagerAgent
from agents.sentiment_agent import SentimentAgent
from agents.contrarian_agent import ContrarianAgent
from agents.graph import DebateGraph, run_debate
from functools import partial
from utils.freshness import CRYPTO5_SEMANTICS_PATH, CRYPTO5_PROCESSED as _DIR, CRYPTO5_DEBATE_ASSETS as _ASSETS

fm_agent = FundManagerAgent(semantics_path=CRYPTO5_SEMANTICS_PATH)
graph = DebateGraph(
    MacroAgent(assets=_ASSETS, data_dir=_DIR), TechnicalAgent(), RiskAgent(data_dir=_DIR),
    fm_agent, SentimentAgent(macro_assets={"macro"}, fear_greed_assets=set(_ASSETS)),
    ContrarianAgent(regime_stream_df=regime_stream),
)
picks = _select_timestamps(regime_stream, "BTC_USDT", 24, True)
t0 = time.time()
result = partial(run_debate, semantics_path=CRYPTO5_SEMANTICS_PATH)(
    picks[0][0], "BTC_USDT", regime_stream, aligned_dfs, graph
)
elapsed = time.time() - t0
print(f"Single debate OK in {elapsed:.1f}s -> {result.fund_manager_decision.final_action} "
      f"(conviction={result.fund_manager_decision.conviction:.3f})")
print(f"Estimated total for {len(picks)} debates: {elapsed * len(picks) / 3600:.1f}h")

Data OK: 334355 regime rows, ['BTC_USDT', 'ETH_USDT', 'BNB_USDT', 'LTC_USDT', 'XRP_USDT']


In [ ]:
!cd /kaggle/working/marta && PYTHONPATH=/kaggle/working/marta python scripts/kaggle_debate_coverage.py --stride-hours 24 --all-regimes --asset BTC_USDT